<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Guida Completa ed Intuitiva al Filtro a Istogramma (Histogram Filter)
### La stima discreta dello stato per il mantenimento della corsia stradale (Lane Filter nel Duckiebot)

Benvenuto nel terzo notebook della serie sulla localizzazione probabilistica! Nei notebook precedenti abbiamo esplorato due approcci diversi:
1. Il **Filtro di Kalman (KF)**: descrive l'incertezza con una formula matematica continua (media $\mu$ e matrice di covarianza $\Sigma$), ma richiede per forza che il mondo sia lineare e gaussiano.
2. Il **Filtro a Particelle (PF)**: usa un campionamento casuale Monte Carlo con particelle libere nello spazio continuo.

### Perché serve il Filtro a Istogramma (Histogram Filter)?
In molte applicazioni reali di robotica mobile e guida autonoma, non abbiamo bisogno di calcolare integrali complessi o di campionare particelle a caso:
* Possiamo **suddividere lo spazio degli stati in una griglia fissa di caselle (bin)**, esattamente come una scacchiera o un istogramma 2D!
* Ogni casella della griglia rappresenta un'ipotesi discreta (il suo baricentro o centroide).
* A ogni casella assegniamo una **probabilità (o massa di probabilità $w^i_t$)**, con la condizione che la somma di tutte le caselle sia pari a $1$:
  $$\sum_{i=1}^N w_t^i = 1$$
* Matematicamente, questa distribuzione discreta viene descritta come somma pesata di funzioni delta di Dirac:
  $$bel(\mathbf{x}_t) = \sum_{i=1}^N w_t^i \delta(\mathbf{x}_t - \mathbf{x}^i)$$

### L'obiettivo in Duckietown: Il Lane Filter (Filtro di Corsia)
Nel Duckiebot, l'Histogram Filter è il "cervello" che consente la **guida autonoma all'interno della corsia (Lane Following)**.
Invece di stimare la posizione assoluta nel mondo $(x, y)$, stimiamo la **posa relativa del robot rispetto alla corsia stradale**, descritta da sole due variabili:
* **$d$ (spostamento laterale):** la distanza del Duckiebot dal centro esatto della corsia destra (in metri).
* **$\phi$ (angolo di rotta o orientamento):** l'angolo del Duckiebot rispetto alla direzione della corsia (in radianti).

L'interfaccia (API) del filtro si compone di tre funzioni fondamentali:
1. **`prior()`**: inizializza la griglia delle probabilità $w_0$.
2. **`predict()`**: propaga in avanti le masse sulla griglia usando i dati degli encoder delle ruote (odometria) e applica una sfocatura gaussiana per modellare il rumore.
3. **`update()`**: aggiorna le probabilità usando le linee della strada (bianche e gialle) rilevate dalla telecamera attraverso un geniale algoritmo di voto geometrico!

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import cv2
import yaml
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.stats import multivariate_normal
from math import floor, sqrt

print("Librerie importate con successo!")

---
# Passo 1: Definizione dello Stato della Corsia: $[d, \phi]$

Nel sistema di navigazione di Duckietown, il robot viaggia nella corsia di destra.
Lo stato che vogliamo stimare istante per istante è la coppia:

$$\mathbf{x} = \begin{bmatrix} d \\ \phi \end{bmatrix}$$

<p style="text-align: center">
<img src="../../assets/images/state.png" alt="Stato Lane Filter" width="60%">
</p>

### Significato semantico delle due variabili:
1. **$d$ (Distanza dal centro corsia):**
   * $d = 0$: il Duckiebot è perfettamente centrato nella sua corsia di marcia.
   * $d > 0$: il robot è spostato verso sinistra (verso la striscia continua gialla di mezzeria).
   * $d < 0$: il robot è spostato verso destra (verso la striscia tratteggiata bianca del bordo strada).
2. **$\phi$ (Angolo relativo alla corsia):**
   * $\phi = 0$: il robot punta dritto in avanti, parallelo alle linee della strada.
   * $\phi > 0$: il robot sta sterzando verso sinistra (in senso antiorario).
   * $\phi < 0$: il robot sta sterzando verso destra (in senso orario).

In [ ]:
# Carichiamo l'immagine reale scattata dalla telecamera del Duckiebot
img_bgr = cv2.imread("../../assets/images/pic1.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 5))
plt.imshow(img_rgb)
plt.title("Vista dalla Telecamera del Duckiebot (pic1.png)", fontsize=13)
plt.axis("off")
plt.show()

---
# Passo 2: La Configurazione e la Creazione della Griglia 2D

I parametri del filtro (intervallo di ricerca, larghezza della strada, risoluzione degli encoder) sono definiti nel file di configurazione `assets/histogram_filter_config.yaml`.

Carichiamo questi parametri e creiamo la nostra griglia di campionamento bidimensionale (`np.mgrid`) per $d$ e per $\phi$:

In [ ]:
# Carichiamo i parametri dal file YAML
with open("../../assets/histogram_filter_config.yaml", "r") as stream:
    params = yaml.safe_load(stream)

hp = params["lane_filter_histogram_configuration"]

# Creiamo la griglia discreta (Meshgrid 2D)
d_grid, phi_grid = np.mgrid[
    hp["d_min"] : hp["d_max"] : hp["delta_d"],
    hp["phi_min"] : hp["phi_max"] : hp["delta_phi"]
]

grid_spec = {
    "d": d_grid,
    "phi": phi_grid,
    "delta_d": hp["delta_d"],
    "delta_phi": hp["delta_phi"],
    "d_min": hp["d_min"],
    "d_max": hp["d_max"],
    "phi_min": hp["phi_min"],
    "phi_max": hp["phi_max"],
}

robot_spec = {
    "wheel_radius": hp["wheel_radius"],
    "encoder_resolution": hp["encoder_resolution"],
    "wheel_baseline": hp["wheel_baseline"],
}

road_spec = {
    "linewidth_white": hp["linewidth_white"],
    "linewidth_yellow": hp["linewidth_yellow"],
    "lanewidth": hp["lanewidth"],
}

cov_mask = [hp["sigma_d_mask"], hp["sigma_phi_mask"]]
mean_0 = [hp["mean_d_0"], hp["mean_phi_0"]]
cov_0 = [[hp["sigma_d_0"]**2, 0.0], [0.0, hp["sigma_phi_0"]**2]]

# Inizializziamo l'array delle credenze (belief) vuoto con la stessa forma della griglia
belief = np.empty(d_grid.shape)
print(f"Dimensioni della Griglia dell'Istogramma: {belief.shape} caselle")
print(f"Risoluzione: delta_d = {hp['delta_d']} m, delta_phi = {hp['delta_phi']} rad")

---
# Passo 3: Inizializzazione della Credenza — `histogram_prior`

All'inizio, prima ancora che il robot si muova o legga immagini, dobbiamo assegnare a ciascuna cella della griglia una probabilità iniziale $w_0$.
Assumiamo che il Duckiebot sia posizionato all'incirca al centro della corsia ($d=0$) e allineato in avanti ($\phi=0$).

Costruiamo una distribuzione gaussiana 2D centrata in $[0, 0]$ e ne calcoliamo il valore in corrispondenza del baricentro di ciascuna casella:
$$w_0^i = \frac{1}{2\pi \sigma_d \sigma_\phi} \exp\left( -\frac{d^2}{2\sigma_d^2} - \frac{\phi^2}{2\sigma_\phi^2} \right)$$

In [ ]:
def histogram_prior(belief, grid_spec, mean_0, cov_0):
    """
    Inizializza la griglia dell'istogramma campionando una densita gaussiana 2D.
    """
    pos = np.empty(belief.shape + (2,))
    pos[:, :, 0] = grid_spec["d"]
    pos[:, :, 1] = grid_spec["phi"]
    
    RV = multivariate_normal(mean_0, cov_0)
    belief = RV.pdf(pos)
    # Normalizziamo affinche la somma di tutte le caselle faccia 1
    belief = belief / np.sum(belief)
    return belief

# Inizializziamo e visualizziamo il Prior
belief = histogram_prior(belief, grid_spec, mean_0, cov_0)

plt.figure(figsize=(7, 6))
plt.imshow(belief, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
plt.colorbar(label="Massa di Probabilita")
plt.title("Prior Belief Iniziale bel(x_0)", fontsize=13)
plt.xlabel("Orientamento phi [rad]", fontsize=11)
plt.ylabel("Distanza dal centro corsia d [m]", fontsize=11)
plt.show()

---
# Passo 4: La Predizione del Movimento — `histogram_predict`

Quando il Duckiebot si muove, i sensori ottici delle ruote (encoder) contano quanti "scatti" (*ticks*) hanno fatto la ruota sinistra e la ruota destra:
* $R$: raggio della ruota ($0.0318\text{ m}$).
* $\alpha = 2\pi / \text{resolution}$: angolo per tick.
* Spostamento ruota sinistra: $d_L = R \cdot \alpha \cdot \text{ticks}_L$.
* Spostamento ruota destra: $d_R = R \cdot \alpha \cdot \text{ticks}_R$.

Dalla cinematica a guida differenziale (*Differential Drive*):
1. **Avanzamento longitudinale lungo la direzione del robot:**
   $$d_A = \frac{d_L + d_R}{2}$$
2. **Rotazione angolare:**
   $$w = \frac{d_R - d_L}{L_{\text{baseline}}}$$
3. **Spostamento laterale rispetto alla corsia:**
   Se il robot viaggia con orientamento $\phi_t$, la componente laterale è:
   $$v = d_A \sin(\phi_t)$$

### Cosa fa la funzione `histogram_predict` sulla griglia?
1. Prende la massa presente in ciascuna casella $(i, j)$ e ne calcola la nuova posizione dopo il moto: $d_{\text{new}} = d + v$, $\phi_{\text{new}} = \phi + w$.
2. Calcola i nuovi indici di casella $(i_{\text{new}}, j_{\text{new}})$ e accumula la massa.
3. **Aggiunta del rumore di processo:** Nel mondo reale le ruote slittano! Questo rumore viene modellato applicando un filtro di **sfocatura gaussiana 2D (`gaussian_filter`)** sull'istogramma, che disperde dolcemente la massa nelle caselle adiacenti esattamente come farebbe la covarianza $+Q_t$!

In [ ]:
def get_estimate(belief, grid_spec):
    """Calcola la stima media (valore atteso) dello stato d e phi."""
    d_est = np.sum(belief * grid_spec["d"])
    phi_est = np.sum(belief * grid_spec["phi"])
    return d_est, phi_est


def histogram_predict(belief, left_encoder_ticks, right_encoder_ticks, grid_spec, robot_spec, cov_mask):
    """
    Propaga la credenza sulla griglia usando i dati degli encoder delle ruote.
    """
    belief_in = belief
    
    # 1. Calcolo degli spostamenti dalle ruote
    R = robot_spec["wheel_radius"]
    alpha = 2 * np.pi / robot_spec["encoder_resolution"]
    d_left = R * alpha * left_encoder_ticks
    d_right = R * alpha * right_encoder_ticks
    
    d_A = (d_left + d_right) / 2.0
    w = (d_right - d_left) / robot_spec["wheel_baseline"]
    
    _, phi_curr = get_estimate(belief, grid_spec)
    v = d_A * np.sin(phi_curr)
    
    # Nuove coordinate dei centroidi
    d_t = grid_spec["d"] + v
    phi_t = grid_spec["phi"] + w
    
    p_belief = np.zeros(belief.shape)
    
    # 2. Riassegniamo la massa alle nuove caselle corrispondenti
    for i in range(belief.shape[0]):
        for j in range(belief.shape[1]):
            if belief[i, j] > 0:
                # Scartiamo se esce fuori dai limiti della griglia
                if (
                    d_t[i, j] > grid_spec["d_max"]
                    or d_t[i, j] < grid_spec["d_min"]
                    or phi_t[i, j] < grid_spec["phi_min"]
                    or phi_t[i, j] > grid_spec["phi_max"]
                ):
                    continue
                
                i_new = int(floor((d_t[i, j] - grid_spec["d_min"]) / grid_spec["delta_d"]))
                j_new = int(floor((phi_t[i, j] - grid_spec["phi_min"]) / grid_spec["delta_phi"]))
                
                if 0 <= i_new < belief.shape[0] and 0 <= j_new < belief.shape[1]:
                    p_belief[i_new, j_new] += belief[i, j]
                    
    # 3. Modelliamo il rumore di processo mediante filtro Gaussiano (sfocatura dell'incertezza)
    s_belief = np.zeros(belief.shape)
    gaussian_filter(p_belief, cov_mask, output=s_belief, mode="constant")
    
    if np.sum(s_belief) == 0:
        return belief_in
        
    # Normalizziamo affinche la somma totale rimanga 1
    belief = s_belief / np.sum(s_belief)
    return belief

In [ ]:
# Simuliamo uno scatto di movimento: 10 tick a sinistra e 20 tick a destra (il robot curva)
belief_pred = histogram_predict(belief, left_encoder_ticks=10, right_encoder_ticks=20, 
                                grid_spec=grid_spec, robot_spec=robot_spec, cov_mask=cov_mask)

plt.figure(figsize=(7, 6))
plt.imshow(belief_pred, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
plt.colorbar(label="Massa di Probabilita")
plt.title("Credenza Predetta dopo il Movimento (bar_bel)", fontsize=13)
plt.xlabel("Orientamento phi [rad]", fontsize=11)
plt.ylabel("Distanza dal centro corsia d [m]", fontsize=11)
plt.show()

---
# Passo 5: Il Modello di Misura Visivo — Segmenti, Proiezione e Voti

Ora arriva la parte più affascinante del sistema di percezione del Duckiebot:
1. **Rilevamento Segmenti:** La telecamera individua i segmenti delle strisce stradali (bianchi a destra, gialli a sinistra).
2. **Proiezione al Suolo (Ground Projection):** Tramite la matrice omografica e la calibrazione della telecamera, ogni segmento 2D dell'immagine viene proiettato sul piano 3D del pavimento in metri.
3. **L'Algoritmo di Voto Geometrico (`generate_vote`):**
   * Immagina di aver visto un segmento bianco a terra con estremi $p_1$ e $p_2$.
   * Calcoliamo il vettore tangente $\hat{t}$ e il vettore normale $\hat{n}$:
     $$\hat{t} = \frac{p_2 - p_1}{\|p_2 - p_1\|}, \quad \hat{n} = \begin{bmatrix} -\hat{t}_y \\ \hat{t}_x \end{bmatrix}$$
   * Calcoliamo la distanza normale $d_i$ e l'angolo di inclinazione $\phi_i = \arcsin(\hat{t}_y)$.
   * Correggiamo la distanza $d_i$ tenendo conto della geometria nota della carreggiata:
     * La linea **bianca** si trova sul bordo destro (a distanza $-W_{\text{lane}} / 2$).
     * La linea **gialla** si trova sulla mezzeria (a distanza $+W_{\text{lane}} / 2$).
   * In questo modo, **ogni singolo segmentino vota** per quale posa $[d, \phi]$ il robot dovrebbe avere!

<p style="text-align: center">
<img src="../../assets/images/03-histogram-filter/Votingalgorithm.png" alt="Algoritmo di Voto" width="80%">
</p>

In [ ]:
# Definizione delle strutture dati per i segmenti di linea
class SegmentColor:
    WHITE = 0
    YELLOW = 1
    RED = 2

class SegmentPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def as_array(self):
        return np.array([self.x, self.y])

class Segment:
    def __init__(self, p1, p2, color):
        self.points = [SegmentPoint(p1[0], p1[1]), SegmentPoint(p2[0], p2[1])]
        self.color = color


def prepare_segments(segments):
    """Filtra i segmenti escludendo quelli rossi o situati dietro al robot."""
    filtered = []
    for seg in segments:
        if seg.color not in [SegmentColor.WHITE, SegmentColor.YELLOW]:
            continue
        # Filtra segmenti che stanno dietro al veicolo (x < 0)
        if seg.points[0].x < 0 or seg.points[1].x < 0:
            continue
        filtered.append(seg)
    return filtered


def generate_vote(segment, road_spec):
    """
    Dato un segmento stradale proiettato al suolo, calcola la stima (d, phi)
    votata da quel segmento in base alla corsia stradale standard.
    """
    p1 = segment.points[0].as_array()
    p2 = segment.points[1].as_array()
    diff = p2 - p1
    norm = np.linalg.norm(diff)
    if norm == 0:
        return 0.0, 0.0
        
    t_hat = diff / norm
    n_hat = np.array([-t_hat[1], t_hat[0]])
    
    d1 = np.inner(n_hat, p1)
    d2 = np.inner(n_hat, p2)
    d_i = (d1 + d2) / 2.0
    phi_i = np.arcsin(np.clip(t_hat[1], -1.0, 1.0))
    
    if segment.color == SegmentColor.WHITE:  # Linea bianca del bordo destro
        if p1[0] > p2[0]:
            d_i -= road_spec["linewidth_white"]
        else:
            d_i = -d_i
            phi_i = -phi_i
        d_i -= road_spec["lanewidth"] / 2.0
        
    elif segment.color == SegmentColor.YELLOW:  # Linea gialla di mezzeria (sinistra)
        if p2[0] > p1[0]:
            d_i -= road_spec["linewidth_yellow"]
            phi_i = -phi_i
        else:
            d_i = -d_i
        d_i = road_spec["lanewidth"] / 2.0 - d_i
        
    return d_i, phi_i

---
# Passo 6: Costruzione dell'Istogramma dei Voti — `generate_measurement_likelihood`

Tutti i segmenti rilevati votano per una casella $(d_i, \phi_i)$ della griglia:
* Creiamo una matrice di zeri grande quanto la griglia.
* Per ogni segmento, calcoliamo $(d_i, \phi_i)$.
* Se il voto cade all'interno dei limiti, incrementiamo di $+1$ il conteggio della cella corrispondente.
* Normalizziamo la matrice in modo che la somma faccia $1$. Questo è il nostro **Measurement Likelihood** $p(\mathbf{z}_t | \mathbf{x}^i)$!

In [ ]:
def generate_measurement_likelihood(segments, road_spec, grid_spec):
    """
    Genera l'istogramma 2D di verosimiglianza accumulando i voti di tutti i segmenti.
    """
    measurement_likelihood = np.zeros(grid_spec["d"].shape)
    
    for seg in segments:
        d_i, phi_i = generate_vote(seg, road_spec)
        
        # Ignora i voti fuori dai bordi dell'istogramma
        if (
            d_i > grid_spec["d_max"]
            or d_i < grid_spec["d_min"]
            or phi_i < grid_spec["phi_min"]
            or phi_i > grid_spec["phi_max"]
        ):
            continue
            
        i = int(floor((d_i - grid_spec["d_min"]) / grid_spec["delta_d"]))
        j = int(floor((phi_i - grid_spec["phi_min"]) / grid_spec["delta_phi"]))
        
        if 0 <= i < measurement_likelihood.shape[0] and 0 <= j < measurement_likelihood.shape[1]:
            measurement_likelihood[i, j] += 1.0
            
    if np.sum(measurement_likelihood) == 0:
        return None
        
    measurement_likelihood /= np.sum(measurement_likelihood)
    return measurement_likelihood

---
# Passo 7: L'Aggiornamento Bayesiano — `histogram_update`

L'aggiornamento bayesiano su una griglia discreta è di una semplicità disarmante:
$$bel(\mathbf{x}_t) = \frac{\bar{w}_t^i \cdot p(\mathbf{z}_t | \mathbf{x}^i)}{\sum_j \bar{w}_t^j \cdot p(\mathbf{z}_t | \mathbf{x}^j)}$$

In Python e NumPy, questo corrisponde a una semplice **moltiplicazione elemento per elemento** (`np.multiply`) tra la matrice del Prior e la matrice dei Voti della telecamera, seguita da una normalizzazione per dividere per la somma!

In [ ]:
def histogram_update(belief, segments, road_spec, grid_spec):
    """
    Aggiorna la credenza moltiplicando il prior per la verosimiglianza delle misure.
    """
    segments_filtered = prepare_segments(segments)
    measurement_likelihood = generate_measurement_likelihood(segments_filtered, road_spec, grid_spec)
    
    if measurement_likelihood is not None:
        # Fusione Bayesiana: moltiplicazione elemento per elemento (Hadamard product)
        belief = np.multiply(belief, measurement_likelihood)
        
        # Normalizzazione
        total_mass = np.sum(belief)
        if total_mass == 0:
            belief = measurement_likelihood
        else:
            belief /= total_mass
            
    return measurement_likelihood, belief

---
# Passo 8: Test del Ciclo Completo e Visualizzazione Finale

Creiamo un insieme di segmenti rilevati coerenti con la nostra corsia (segmenti bianchi a destra e gialli a sinistra) ed eseguiamo l'**Update** a partire dalla credenza predetta:
* Visualizziamo il **Measurement Likelihood** (cosa vede la telecamera).
* Visualizziamo il **Belief Finale Aggiornato** (la sintesi probabilistica ottimale).

In [ ]:
# Creiamo segmenti realistici di test proiettati al suolo
# 1. Linea bianca a destra (x tra 0.2 e 0.8 metri davanti al robot, y a circa -0.11 m)
seg_white_1 = Segment([0.25, -0.11], [0.45, -0.11], SegmentColor.WHITE)
seg_white_2 = Segment([0.55, -0.11], [0.75, -0.11], SegmentColor.WHITE)

# 2. Linea gialla a sinistra (x tra 0.2 e 0.7 metri davanti, y a circa +0.12 m)
seg_yellow_1 = Segment([0.20, 0.12], [0.40, 0.12], SegmentColor.YELLOW)
seg_yellow_2 = Segment([0.45, 0.12], [0.65, 0.12], SegmentColor.YELLOW)

test_segments = [seg_white_1, seg_white_2, seg_yellow_1, seg_yellow_2]

# Eseguiamo l'aggiornamento
meas_like, belief_final = histogram_update(belief_pred, test_segments, road_spec, grid_spec)

# Visualizziamo i risultati affiancati
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Predizione da Odometria
im1 = axes[0].imshow(belief_pred, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
axes[0].set_title("1. Predizione Odometria (bar_bel)", fontsize=13)
axes[0].set_xlabel("phi [rad]")
axes[0].set_ylabel("d [m]")
plt.colorbar(im1, ax=axes[0])

# 2. Verosimiglianza Telecamera (Voti)
im2 = axes[1].imshow(meas_like, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
axes[1].set_title("2. Voti Segmenti Telecamera (Likelihood)", fontsize=13)
axes[1].set_xlabel("phi [rad]")
axes[1].set_ylabel("d [m]")
plt.colorbar(im2, ax=axes[1])

# 3. Posteriore Fuso Definitivo
im3 = axes[2].imshow(belief_final, origin="lower", cmap="jet", extent=[hp["phi_min"], hp["phi_max"], hp["d_min"], hp["d_max"]], aspect="auto")
axes[2].set_title("3. Credenza Finale Aggiornata bel(x_t)", fontsize=13)
axes[2].set_xlabel("phi [rad]")
axes[2].set_ylabel("d [m]")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

d_final, phi_final = get_estimate(belief_final, grid_spec)
print(f"Stima Ottimale Finale della Posa nella Corsia:")
print(f" -> Distanza dal centro (d):   {d_final:.4f} metri")
print(f" -> Angolo di orientamento (phi): {phi_final:.4f} radianti ({np.degrees(phi_final):.2f} gradi)")

---
# Tabella di Sintesi del Lane Filter a Istogramma

| Componente | Significato Fisico | Come viene Calcolato in Python |
| :--- | :--- | :--- |
| **$d$** | Distanza dal centro corsia | Asse verticale della griglia ($-0.3\text{ m} \dots +0.3\text{ m}$) |
| **$\phi$** | Angolo di rotta rispetto alla strada | Asse orizzontale della griglia ($-1.5\text{ rad} \dots +1.5\text{ rad}$) |
| **`histogram_prior`** | Conoscenza a priori iniziale | Campionamento di normale 2D multivariata centrata in $(0, 0)$ |
| **`histogram_predict`** | Avanzamento del robot dalle ruote | Integrazione tick encoder + convoluzione con `gaussian_filter` |
| **`generate_vote`** | Stima posa fornita da una riga bianca/gialla | Geometria vettoriale: vettore normale, tangente e larghezza corsia |
| **`generate_measurement_likelihood`** | Mappa dei voti della telecamera | Istogramma 2D con accumulo $+1$ per ogni segmento valido |
| **`histogram_update`** | Fusione ottima tra odometria e visione | Prodotto elemento per elemento (`np.multiply`) + normalizzazione |

---

### Prossimo Passo: L'Extended Kalman Filter (EKF)
Nel prossimo ed ultimo notebook affronteremo il culmine della localizzazione robotica: **[EKF Localization](../04-ekf-localization/ekf_localization.ipynb)**, dove utilizzeremo i landmark (AprilTag) e le matrici Jacobiane per linearizzare in tempo reale la navigazione del Duckiebot!